**CSP=(X,D,C)**

CSP (Constraint Satisfaction Problem) Framework = (variable, domain, constraint)

In [1]:
from src.csp import CSP
from src.constraints import BinaryConstraint
from src.solvers import (
    backtracking_search,
    backtracking_search_mrv,
    backtracking_search_mrv_degree,
    backtracking_search_with_stats,
    backtracking_search_mrv_with_stats,
    backtracking_search_mrv_degree_with_stats,
)
from src.generators import (
    generate_graph_coloring_csp
)

<h1> Map Coloring Example </h1>

In [2]:
variables = [
    "WA",
    "NT",
    "SA",
    "Q",
]

colors = [
    "Red",
    "Green",
    "Blue",
]

domains = {
    variable: colors.copy()
    for variable in variables
}

constraints = [

    BinaryConstraint(
        "WA",
        "NT",
    ),

    BinaryConstraint(
        "WA",
        "SA",
    ),

    BinaryConstraint(
        "NT",
        "SA",
    ),

    BinaryConstraint(
        "NT",
        "Q",
    ),

    BinaryConstraint(
        "SA",
        "Q",
    ),
]

neighbors = {

    "WA": [
        "NT",
        "SA",
    ],

    "NT": [
        "WA",
        "SA",
        "Q",
    ],

    "SA": [
        "WA",
        "NT",
        "Q",
    ],

    "Q": [
        "NT",
        "SA",
    ],
}

In [3]:
problem = CSP(
    variables,
    domains,
    constraints,
    neighbors,
)

For this step, we will create the neighbors manually, but a better design is to have CSP itself create this graph later based on the constraints.

In [4]:
problem.validate()

problem.get_domains_copy()

{'WA': ['Red', 'Green', 'Blue'],
 'NT': ['Red', 'Green', 'Blue'],
 'SA': ['Red', 'Green', 'Blue'],
 'Q': ['Red', 'Green', 'Blue']}

In [5]:
assignment = {
    "WA":"Red",
    "NT":"Green",
    "SA":"Blue",
}

In [6]:
problem.is_consistent(
    assignment
)

True

In [7]:
assignment = {
    "WA":"Red",
    "NT":"Red",
}

In [8]:
problem.is_consistent(
    assignment
)

False

Naive Backtracking

In [9]:
solution = backtracking_search(
    problem
)

solution

{'WA': 'Red', 'NT': 'Green', 'SA': 'Blue', 'Q': 'Red'}

<h1> MRV — Minimum Remaining Values </h1>

Idea:
Choose the variable that has the fewest remaining choices.

In [10]:
solution_basic = backtracking_search(
    problem
)

solution_basic

{'WA': 'Red', 'NT': 'Green', 'SA': 'Blue', 'Q': 'Red'}

In [11]:
solution_mrv = backtracking_search_mrv(
    problem
)

solution_mrv

{'WA': 'Red', 'NT': 'Green', 'SA': 'Blue', 'Q': 'Red'}

<h1> Degree Heuristic + MRV Tie Breaking </h1>

Degree Idea 

Choose the variable that:</br>
has the most constraints with other variables.

High impact variable first

**MRV + Degree combination**

Algorithm:

First MRV: Lowest Domain.

If multiple options are equal: Highest Degree.

In [12]:
solution_mrv_degree = backtracking_search_mrv_degree(
    problem
)

solution_mrv_degree

{'NT': 'Red', 'SA': 'Green', 'WA': 'Blue', 'Q': 'Blue'}

In CSP, each Assignment is a Node in the Search Tree.

In [13]:
solution, stats = backtracking_search_with_stats(
    problem
)

solution

{'WA': 'Red', 'NT': 'Green', 'SA': 'Blue', 'Q': 'Red'}

In [14]:
stats.summary()

{'Nodes Visited': 5, 'Assignments Tried': 7, 'Backtracks': 0}

We still don't see the power of heuristics, so let's build a real benchmark.

<h3> Instrumenting Heuristic Solvers </h3>

Comparing:

Naive Backtracking</br>
vs</br>
MRV Backtracking</br>
vs</br>
MRV + Degree

Metric:

Nodes Visited</br>
Assignments Tried</br>
Backtracks</br>

In [15]:
solution_mrv_stats, stats_mrv = backtracking_search_mrv_with_stats(problem)

solution_mrv_stats

{'WA': 'Red', 'NT': 'Green', 'SA': 'Blue', 'Q': 'Red'}

In [16]:
stats_mrv.summary()

{'Nodes Visited': 5, 'Assignments Tried': 7, 'Backtracks': 0}

Heuristics show their value when the search space is large or the constraints are hard.

In [17]:
solution_mrv_degree_stats, stats_mrv_degree = (
    backtracking_search_mrv_degree_with_stats(
        problem
    )
)

solution_mrv_degree_stats

{'NT': 'Red', 'SA': 'Green', 'WA': 'Blue', 'Q': 'Blue'}

In [18]:
stats_mrv_degree.summary()

{'Nodes Visited': 5, 'Assignments Tried': 9, 'Backtracks': 0}

That is, Degree has an effect on the order of selection, but because the problem is simple, we have not yet entered an area where a real difference can be seen.

In [19]:
random_problem = generate_graph_coloring_csp(
    num_variables=20,
    num_colors=4,
    edge_probability=0.2,
)

random_problem

In [22]:
len(random_problem.variables)

20

In [23]:
len(random_problem.constraints)

42

In [24]:
random_problem.neighbors["X0"]

['X4', 'X5', 'X14', 'X17', 'X18']

<h1> Benchmark Framework </h1>